In [ ]:
using DelimitedFiles;   #Paquetería para leer y escribir datos
using CairoMakie;       #Paquetería para graficar datos
using Printf; 
using FFTW;             #Paquetería para realizar análisis de Fourier
using LaTeXStrings;     #Paquetería para emplear texto en LaTeX en las gráficas
using LsqFit;           #Paquetería para realizar un ajuste por mínimos cuadrados
using Statistics;       #Paquetería para realizar cálculos de estadística básica

#Definimos la función que calcula nuestro aproximante estadístico
AproxLambda(NSides) = (2π / (1 - cos(2π / NSides)));

DataPath = "Quasiperiodic-Tiles/Global Structural Studies/Data/SI_Fig6-A_SigmaSquare_1D/"; #Ruta de acceso a los datos para analizar

In [ ]:
#Valores del índice asociado a la tesela "grande" que determina la longitud de escala en el análisis de Fourier de las gráficas de Sigma^2.
#Los valores a partir de la llave 101 se corresponden a la tesela más grande debido a que no tenemos datos suficientes para un buen análisis de Fourier
Dict_Index_PseudoPeriodo = Dict(
                                5   => 1,
                                7   => 0,
                                9   => 4,
                                11  => 1,
                                13  => 0,
                                15  => 7,
                                17  => 1,
                                19  => 9,
                                21  => 10,
                                23  => 11,
                                25  => 1,
                                27  => 12,
                                29  => 0,
                                31  => 14,
                                33  => 15,
                                35  => 0,
                                37  => 16,
                                39  => 18,
                                41  => 20,
                                43  => 21,
                                45  => 20,
                                47  => 23,
                                49  => 23,
                                51  => 1,
                                101 => 0,
                                201 => 0,
                                401 => 0,
                                801 => 0,
);

### Carga de datos

In [ ]:
#Datos de las vecindades generadas al obtener los datos
NSides = 29;        #Simetría rotacional de los sistemas cuasperiódicos a analizar
Radio = 1800;       #Radio de las vecindades circulares
ΔStep = 0.05;       #Salto en los valores de R
Notebooks = 10;     #Número de notebooks generados en el servidor
Vecindades = 10000; #Número de vecindades locales generadas por cada notebook

#Datos de la Sigma^2
NR = vec(readdlm(DataPath * "NR_N$(NSides)_ThetaStarVectors0_Acumulados$(Vecindades)_DeltaStep0P05_Radius$(Radio)_Nb1.csv"));
NR2 = vec(readdlm(DataPath * "NR2_N$(NSides)_ThetaStarVectors0_Acumulados$(Vecindades)_DeltaStep0P05_Radius$(Radio)_Nb1.csv"));
for Nb in 2:Notebooks
    NR .+= vec(readdlm(DataPath * "NR_N$(NSides)_ThetaStarVectors0_Acumulados$(Vecindades)_DeltaStep0P05_Radius$(Radio)_Nb$(Nb).csv"));
    NR2 .+= vec(readdlm(DataPath * "NR2_N$(NSides)_ThetaStarVectors0_Acumulados$(Vecindades)_DeltaStep0P05_Radius$(Radio)_Nb$(Nb).csv"));
end
NR ./= (Vecindades * Notebooks);
NR2 ./= (Vecindades * Notebooks);
σ2 = NR2 .- (NR .^ 2);

#Generamos el intervalo con los valores de la R asociados a los datos de sigma cuadrada (Incluye factor de 2*sqrt(π*Rho)
#necesario para mantener densidad de puntos constantes en decorado, independientemente de la simetría rotacional)
R = ΔStep:ΔStep:Radio;

### Gráfica de hiperuniformidad

In [ ]:
# --- Definición de las características del lienzo y las subgráficas en él ---
Fig = Figure(size = (1800, 800));   #Lienzo en blanco donde se graficara
Radio_Viz = 1810;                   #Máximo radio al cual se visualizará la gráfica
λ_Inf_Ax = Axis(
                Fig[1, 1],                                                   #Posición en el lienzo donde se realizará la gráfica
                title = L"N = %$(NSides)",                                   #Título de la gráfica
                xlabel = L"R",                                               #Etiqueta que aparece en el eje horizontal
                ylabel = L"\sigma^{2}(R)",                                   #Etiqueta que aparece en el eje vertical
                titlesize = 55,                                              #Tamaño del título
                xlabelsize = 55,                                             #Tamaño de la etiqueta al eje horizontal
                ylabelsize = 55,                                             #Tamaño de la etiqueta al eje vertical
                xticklabelsize = 40,                                         #Tamaño para el eje X
                yticklabelsize = 40,                                         #Tamaño para el eje Y
                xticksize = 25,                                              #Tamaño de los ticks horizontales
                yticksize = 25,                                              #Tamaño de los ticks verticales
                limits = ((ΔStep, Radio_Viz), nothing),                      #Límites de la visualización para la gráfica
                ytickformat = values -> [@sprintf("%.0f", v) for v in values],
                #xscale = log10,
                #yscale = log10
               )
hidespines!(λ_Inf_Ax, :t, :r); #Remueve las líneas de la caja que rodea a la gráfica ':t' = top, ':r' = right
hidedecorations!(
                 λ_Inf_Ax,
                 label = false,           #Se oculta o no las etiquetas a los ejes
                 ticklabels = false,      #Se oculta o no los valores de los ticks de los ejes
                 ticks = false            #Se oculta o no los ticks de los ejes
                )
# --- Gráfica de los datos de la σ^2(R) ---
Start = 1; #Dato inicial a partir del cual se comienza a realizar la gráfica
Final = 0; #Datos del final a eliminar (por errores en inconsistencias de tamaño en vecindades)
if NSides == 25
    Final = 50;
    elseif NSides == 29 || NSides == 45
    Final = 170;
end
lines!(λ_Inf_Ax, R[Start:(end - Final)], σ2[Start:(end - Final)])
# --- Gráfica del intervalo donde se calculará Λ_Inf ---
Index = Dict_Index_PseudoPeriodo[NSides]; #Índice para la longitud de escala λ_N
λ = abs(cos(2*Index*π / NSides) * (NSides / 2)); #Escala de longitud de la prototesela con índice (Index + 1)
vlines!(λ_Inf_Ax, [λ, Int(floor(R[end - Final]/λ)) * λ],
        color = :black,
        linestyle = :dash,
        linewidth = 5,
        alpha = 0.75
       )
# --- Cálculo de Λ_Inf ---
Start_λ = findfirst(x -> x >= λ, R);
End_λ = findfirst(x -> x >= Int(floor(R[end - Final]/λ))*λ, R);
R_Acotado = R[Start_λ:End_λ];
Sigma2_Acotado = σ2[Start_λ:End_λ];
#Cálculo de los estadísticos sobre \sigma^2
Media = mean(Sigma2_Acotado);
STD_Media = std(Sigma2_Acotado);
println("El valor promedio de los datos es: $(Media)")
println("La desviación estándar con respecto al promedio es: $(STD_Media)")
# --- Gráfica de Λ_Inf ---
lines!(λ_Inf_Ax, [R[Start_λ], R[End_λ]], [Media, Media],
       color = :red
      )
# --- Guardamos la gráfica ---
Fig

### Gráfica de $\Lambda_{\infty}$ como función de N con barras de error

In [ ]:
#Modelo 2: PROMEDIO
#Starting Point = λ_S

Dict_Lambda_Infty_STD = Dict(
                             5   => [0.4871890425666374, 0.13059158809832916],
                             7   => [0.8840180641058734, 0.1774578870739518],
                             9   => [1.6458712949153735, 0.3436479909545268],
                             11  => [1.7320197837215816, 0.3248856700174585],
                             13  => [2.057109014970975, 0.2783711318839162],
                             15  => [2.990862377703204, 0.5913074791501061],
                             17  => [4.381346896615316, 1.0068257456610512],
                             19  => [5.369806817174149, 1.2002527624932213],
                             21  => [6.96058975323639, 1.3792141781982366],
                             23  => [7.4623008777295885, 1.5304314758902366],
                             25  => [5.824485873513508, 0.9935072867786647],
                             27  => [6.811778504374358, 0.9563938454862345],
                             29  => [10.0294205337879, 1.8098424519494938],
                             31  => [9.841057604982117, 1.4396643721498665],
                             33  => [13.669788613951384, 2.3112159935966017],
                             35  => [13.999235953926256, 2.2761502356443057],
                             37  => [15.23925779568314, 2.4905806056705786],
                             39  => [15.273451668659067, 2.5774564662856907],
                             41  => [16.059440699133848, 2.33060484623209],
                             43  => [17.053208027959023, 2.5273311956542246],
                             45  => [18.32158517727779, 2.5299107662447793],
                             47  => [18.524271451642573, 2.5473709405121476],
                             49  => [20.904799060700515, 2.8976152675758744],
                             51  => [23.417094533666766, 3.2517150228794605],
                             101 => [58.9508557812032, 5.443735405126623],
                             201 => [182.45854042280644, 27.726500547239848],
                             401 => [411.5973167515615, 62.22630840122559],
                             801 => [1095.1218705512401, 205.52659425484615]
                            );

In [ ]:
# --- Definimos el lienzo general y el eje donde se graficará la primera imagen en el lienzo completo ---
fig = Figure(size = (1800, 600), figure_padding = (10, 20, 10, 5)); #Para el size (Alto, Ancho); #Para el padding (izq, der, abajo, arriba)
LambdaInfty_N_1D = Axis(fig[1,1],                       #Posición relativa de la gráfica dentro del lienzo completo de la figura
                        #yscale = log10,                #Escala logarítmica para el eje vertical
                        #xscale = log10,                #Escala logarítmica para el eje horizontal
                        xlabel = L"N",                  #Etiqueta para el eje horizontal
                        ylabel = L"\Lambda_{\infty}",   #Etiqueta para el eje vertical
                        titlesize = 55,                 #Tamaño del título
                        xlabelsize = 55,                #Tamaño de la etiqueta al eje horizontal
                        ylabelsize = 55,                #Tamaño de la etiqueta al eje vertical
                        xticklabelsize = 40,            #Tamaño para el eje X
                        yticklabelsize = 40,            #Tamaño para el eje Y
                        xticksize = 25,                 #Tamaño de los ticks horizontales
                        yticksize = 25,                 #Tamaño de los ticks verticales
                        rightspinevisible = false,      #Oculta la línea vertical derecha
                        topspinevisible = false,        #Oculta la línea horizontal superior
                        xgridvisible = false,           #Mallado en los ejes horizontales
                        ygridvisible = false,           #Mallado en los ejes verticales
                       );
# --- Ajuste por mínimos cuadrados a los datos de Λ_Inf y sus barras de error ---
N_Array = 5:2:51;
Lambdas_Array = [Dict_Lambda_Infty_STD[NSides][1] for NSides in N_Array];
STD_Array = [Dict_Lambda_Infty_STD[NSides][2] for NSides in N_Array];

@. model(x, p) = p[1] + p[2] * (x^p[3])
p0 = [0.5, 0.5, 0.5]
fit = curve_fit(model, N_Array, Lambdas_Array, p0)
K = coef(fit)
println(K)
# --- Gráfica de las Λ_Inf y sus barras de error con la recta que mejor ajusta ---
lines!(LambdaInfty_N_1D, N_Array, K[1] .+ (K[2] .* (N_Array .^ K[3])),
       label = L"\Lambda_{\infty} = %$(round(K[1], digits = 3)) + %$(round(K[2], digits = 3)) N^{%$(round(K[3], digits = 3))}")
scatter!(LambdaInfty_N_1D, N_Array, Lambdas_Array)
errorbars!(LambdaInfty_N_1D, N_Array, Lambdas_Array, STD_Array, 
           whiskerwidth = 10,  #Ancho de la línea horizontal superior/inferior
           color = :black,
           linewidth = 1.5
          )
# --- Guardamos la gráfica ---

fig